# Phase 3A - Abstention Dataset Preparation (Kaggle)

Notebook này lấy 587 câu `heldout_reserve`, chạy retrieval BGE-M3 + BGE-large, tạo review queue 200 case và dừng để human review. Không gọi generator hoặc judge.

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='c53184ad8046ccf9fca0216d01234125d12bdb12'
HF_ARTIFACT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
HF_ARTIFACT_REVISION='locked-bge-m3-512-64-deduplicated-v2'
HF_ARTIFACT_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
HF_ARTIFACT_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'
HF_PREPARATION_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
HF_PREPARATION_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
HF_PREPARATION_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
HF_PREPARATION_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'
AUTHORED_CASES_PATH=''  # Upload reviewed external/counterfactual proposals, then set this path and rerun the build cell.
OVERWRITE_DRAFT=False
SEED=42

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,yaml
from kaggle_secrets import UserSecretsClient
ROOT=Path('/kaggle/working'); PROJECT_ROOT=ROOT/'Text-Mining---NewsQA-RAG'; WORK=ROOT/'phase3_abstention_preparation'
DATA=WORK/'data'; PREP=WORK/'phase2b_preparation'; SOURCE=WORK/'source_retrieval'; DRAFT=WORK/'review_draft'
for path in [DATA,PREP,SOURCE,DRAFT]: path.mkdir(parents=True,exist_ok=True)
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing Phase 3 files'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
secrets=UserSecretsClient()
try: HF_TOKEN=secrets.get_secret('HF_TOKEN') or ''
except Exception: HF_TOKEN=''
assert HF_TOKEN, 'Add the read-only HF_TOKEN Kaggle secret for the private preparation bundle'
os.environ.update({'HF_HOME':str(ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1'})
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def run(args): print('$',' '.join(map(str,args)),flush=True); subprocess.run(list(map(str,args)),cwd=PROJECT_ROOT,check=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})

In [ ]:
from huggingface_hub import hf_hub_download
artifact=DATA/'locked-bge-m3-512-64-deduplicated-v2'
if not (artifact/'bundle_manifest.json').exists():
    bundle=Path(hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=HF_TOKEN or None))
    assert sha(bundle)==HF_ARTIFACT_SHA256; shutil.unpack_archive(bundle,artifact)
if not (PREP/'results/preparation_bundle_manifest.json').exists():
    bundle=Path(hf_hub_download(repo_id=HF_PREPARATION_REPO_ID,repo_type='dataset',revision=HF_PREPARATION_REVISION,filename=HF_PREPARATION_FILENAME,token=HF_TOKEN))
    assert sha(bundle)==HF_PREPARATION_SHA256; shutil.unpack_archive(bundle,PREP)
manifest=json.loads((artifact/'bundle_manifest.json').read_text()); assert manifest['statistics']['resolved_questions']==1152 and manifest['statistics']['chunks']==22766
ids=PREP/'question_ids/heldout_reserve.json'; assert len(json.loads(ids.read_text()))==587
config=PREP/'index/phase2b_config.yaml'; testset=artifact/'testset_resolved.jsonl'; chunks=artifact/'chunks.jsonl'; sparse=artifact/'bge_m3_sparse.pkl'
print('Inputs verified:',artifact,'| reserve questions:',len(json.loads(ids.read_text())))

In [ ]:
source_cases=SOURCE/'source_cases.jsonl'
run([sys.executable,'scripts/create_abstention_source_cases.py','--testset',testset,'--question-ids-file',ids,'--output',source_cases])
run([sys.executable,'scripts/collect_abstention_retrievals.py','--cases',source_cases,'--chunks',chunks,'--sparse-index',sparse,'--config',config,'--run-dir',SOURCE,'--top-k','20','--rerank-top-n','5','--progress'])
command=[sys.executable,'scripts/prepare_abstention_dataset.py','prepare','--mode','compact_200','--locked-root',artifact,'--question-ids-file',ids,'--development-articles','70','--retrievals',SOURCE/'case_retrievals.jsonl','--output-dir',DRAFT]
if AUTHORED_CASES_PATH: command += ['--authored-cases',AUTHORED_CASES_PATH]
if OVERWRITE_DRAFT or any(DRAFT.iterdir()): command += ['--overwrite']
run(command)
draft_manifest=json.loads((DRAFT/'manifest.json').read_text()); print(json.dumps({'observed':draft_manifest['observed'],'deficits':draft_manifest['deficits'],'partitions':draft_manifest['observed_by_partition']},indent=2))
print('Review this file:',DRAFT/'review_queue_readable.json')

## Human-review checkpoint

Lần chạy đầu chưa có authored cases sẽ còn deficit `external_unanswerable` và `counterfactual`. Codex tạo proposals từ queue và nguồn NewsQA bị giữ lại; reviewer điền `human_review`, reviewer độc lập điền `secondary_review`. Sau đó đặt `AUTHORED_CASES_PATH`, chạy lại cell build, review toàn bộ `review_queue_readable.json`, rồi mới dùng lệnh `finalize` trong test plan.

In [ ]:
archive=shutil.make_archive(str(ROOT/'phase3_abstention_preparation_checkpoint'),'zip',WORK)
print('Download checkpoint:',archive,round(Path(archive).stat().st_size/2**20,1),'MiB')